In [ ]:
def build_resnet50(num_classes, img_size=(224, 224), freeze_base=True):
    base_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(*img_size, 3)
    )

    base_model.trainable = not freeze_base
    if freeze_base:
        print(f"🔒 Base frozen: {len(base_model.layers)} layers")

    x = base_model.output
    x = GlobalAveragePooling2D(name="gap")(x)
    x = BatchNormalization(name="bn_1")(x)
    x = Dense(512, activation="relu", name="dense_512")(x)
    x = Dropout(0.4, name="dropout_1")(x)
    x = Dense(256, activation="relu", name="dense_256")(x)
    x = Dropout(0.3, name="dropout_2")(x)
    output = Dense(num_classes, activation="softmax", name="output")(x)

    model = Model(inputs=base_model.input, outputs=output, name="ResNet50_Drug_Classifier")
    return model, base_model

resnet_model, base_model = build_resnet50(NUM_CLASSES)

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

resnet_model.summary()
print(f"\n✅ Total params     : {resnet_model.count_params():,}")
print(f"✅ Trainable params  : {sum([tf.size(w).numpy() for w in resnet_model.trainable_weights]):,}")

In [ ]:
callbacks_phase1 = [
    EarlyStopping(monitor="val_accuracy", patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint("best_resnet_phase1.h5",
                    monitor="val_accuracy", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1)
]

print("🔒 Phase 1: Training only the custom head (base frozen)")
history1 = resnet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=4,
    callbacks=callbacks_phase1
)
print("\\n✅ Phase 1 complete!")

In [ ]:
val_gen.reset()
y_pred  = resnet_model.predict(val_gen, verbose=1)
y_pred_cls = np.argmax(y_pred, axis=1)
y_true_cls = val_gen.classes

resnet_acc = np.mean(y_pred_cls == y_true_cls)
print(f"\\n🎯 ResNet50 Accuracy: {resnet_acc*100:.2f}%")
print(f"\\n{classification_report(y_true_cls, y_pred_cls, target_names=CLASS_NAMES)}")

cm = confusion_matrix(y_true_cls, y_pred_cls)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix — ResNet50", fontsize=13)
plt.ylabel("True"); plt.xlabel("Predicted")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

def predict_drug_image(img_path, top_k=3):
    img   = keras_image.load_img(img_path, target_size=IMG_SIZE)
    arr   = keras_image.img_to_array(img) / 255.0
    arr   = np.expand_dims(arr, axis=0)
    probs = resnet_model.predict(arr, verbose=0)[0]
    top_i = np.argsort(probs)[::-1][:top_k]

    # Show image
    plt.figure(figsize=(4, 4))
    plt.imshow(keras_image.load_img(img_path, target_size=IMG_SIZE))
    plt.title(f"Prediction: {CLASS_NAMES[top_i[0]]}", fontsize=13, fontweight="bold")
    plt.axis("off"); plt.show()

    print(f"\\n📊 Top {top_k} Predictions:")
    print("-" * 42)
    for i, idx in enumerate(top_i, 1):
        bar = "█" * int(probs[idx]*100/5)
        print(f"  {i}. {CLASS_NAMES[idx]:<20} {probs[idx]*100:>6.2f}%  {bar}")
    return CLASS_NAMES[top_i[0]], float(probs[top_i[0]])

# ── Test with a random val image ──
sample_class = random.choice(CLASS_NAMES)
sample_dir   = os.path.join(DATA_DIR, sample_class)
sample_img   = os.path.join(sample_dir, random.choice(os.listdir(sample_dir)))
print(f"True label: {sample_class}")
predict_drug_image(sample_img)

In [ ]:
import pickle
resnet_model.save("resnet50_drug_classifier.h5")
with open("class_names.pkl", "wb") as f: pickle.dump(CLASS_NAMES, f)

print("✅ resnet50_drug_classifier.h5")
print("✅ class_names.pkl")
print("\n🎉 ResNet50 Image Model Done!")
print("   Next → Fusion Model 🔥")